In [1]:
filter_perts = True

In [2]:
import warnings
warnings.filterwarnings(action='ignore', module='pandas')
warnings.filterwarnings(action='ignore', category=FutureWarning, module='ipykernel')
warnings.filterwarnings(action='ignore', category=FutureWarning, module='scanpy')
warnings.filterwarnings(action='ignore', category=FutureWarning, module='anndata')
warnings.filterwarnings(action='ignore', category=UserWarning, module='scanpy')

In [3]:
import os
import json
import requests
from io import StringIO

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

import sys

sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import io
from scLEMBAS import parse_network as pn

In [4]:
n_cores = 30
os.environ["OMP_NUM_THREADS"] = str(n_cores)
os.environ["MKL_NUM_THREADS"] = str(n_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cores)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(n_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(n_cores)

seed = 888
data_path = '/home/hmbaghda/orcd/pool/scLEMBAS/analysis'

author = 'McCauley'

In [5]:
# load from Notebook 01
tf_adata = io.read_tfad(file_name = os.path.join(data_path, 'interim', author + '_tf_activity_all.h5ad'))


cat_col = 'cell_type'
pert_col = 'ligand'
ctrl_pert = 'CTRL'


# filter from Notebook 02
if filter_perts:
    pert_strength = pd.read_csv(os.path.join(data_path, 'processed', author + '_pert_strength_quantification.csv'), 
                               index_col = 0)
    strong_perts = set(pert_strength[pert_strength.strong].perturbation)
    strong_perts.add(ctrl_pert)
    tf_adata = tf_adata[tf_adata.obs[pert_col].isin(strong_perts)].copy()



Load and parse input signaling network:

In [6]:
source_label = 'source_genesymbol'
target_label = 'target_genesymbol'
weight_label = 'mode_of_action'
stimulation_label = 'consensus_stimulation'
inhibition_label = 'consensus_inhibition'

In [7]:
sn_ppis = pn.load_network('omnipath', organism = 'human', static = True)
sn_ppis = pn.correct_network(sn_ppis = sn_ppis,
                                        source_label = source_label, target_label = target_label,
                                        stimulation_label = stimulation_label, inhibition_label = inhibition_label)

/home/hmbaghda/Projects/scLEMBAS/scLEMBAS/parse_network.py:162: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  unique_vals = pd.concat([unique_vals, dup_int], axis = 0)


Add the missing perturbation interactions:

- CHIR99021: GSK3A/B inhibition ([1](https://www.stemcell.com/products/chir99021.html), [Medchem](https://www.medchemexpress.com/laduviglusib.html), [Broad’s Institute Repurposing Hub v08/19/25](https://repo-hub.broadinstitute.org/repurposing#download-data), [Drugbank](https://go.drugbank.com/drugs/DB19676))
- TNFA: TNFRSF1A/B activation ([1](https://www.thermofisher.com/proteins/product/Human-TNF-alpha-Recombinant-Protein/300-01A-50UG))

In [8]:
# # broad repurposing CHIR9902 targets
# url = "https://repo-hub.broadinstitute.org/public/data/repo-drug-annotation-20200324.txt"
# response = requests.get(url, verify=False)  # bypass SSL verification
# response.raise_for_status()  # raise error if failed
# rh = pd.read_csv(StringIO(response.text), sep="\t", comment="!", engine="python")
# rh[rh['pert_iname'] == 'CHIR-99021'].target.tolist()


In [9]:
moa_map = {
    'CHIR99021': {'GSK3A': -1, 'GSK3B': -1, 'CDK1': -1, 'MAPK1': -1}, 
    'TNFA': {'TNFRSF1A': 1, 'TNFRSF1B': 1}
}

# format as input to parsing network
delim = '&'
interactions_to_add = []
moa = []
for source, targets in moa_map.items():
    for target, moa_ in targets.items():
        interactions_to_add.append(delim.join([source, target]))
        moa.append(moa_)
        
        
# add to network
sn_ppis = pn.add_omnipath_interaction(sn_ppis = sn_ppis,
                                      interactions_to_add = interactions_to_add,  
                                      moa = moa,
                                      delim = delim,
                                      source_label = source_label,
                                      target_label = target_label, 
                                      stimulation_label = stimulation_label, 
                                      inhibition_label = inhibition_label
                           )


Filtering the network for interactions with atleast one reference and a curation effort of 2: 

In [10]:
sn_ppis = pn.extract_network(sn_ppis = sn_ppis.copy(), 
                             curation_effort_thresh = 2, 
                             n_references_thresh = 1,
                             resources = 'all', 
                             drop_self = True, 
                             source_label = source_label, 
                             target_label = target_label,
                             verbose = True)

The thresholds filtered 107345  of 133613 interactions


Retain a signaling network of nodes with full paths from input drugs to the inferred TFs:

In [11]:
ligand_labels = sorted(set(tf_adata.obs[pert_col]).difference([ctrl_pert]))
tf_labels = tf_adata.var.index.unique().tolist()

In [12]:
ppi_nodes = set(sn_ppis[source_label].tolist() + sn_ppis[target_label].tolist())
n_tf_intersect_ = len(ppi_nodes.intersection(tf_labels))
print('{} of {} TFs are present in Omnipath upon applying signaling network thresholds'.format(n_tf_intersect_, 
                                                                          tf_adata.shape[1], 
                                                                         ))


423 of 516 TFs are present in Omnipath upon applying signaling network thresholds


In [13]:
n_og = sn_ppis.shape[0]
sn_ppis, ligand_connections = pn.create_connected_network(sn_ppis = sn_ppis, 
                                                       ligand_labels = ligand_labels, 
                                                       tf_labels = tf_labels, 
                                                       source_label = source_label, 
                                                       target_label = target_label,
                                                       path_finder = 'connected')


assert all([len(v) > 0 for k,v in ligand_connections.items()]), 'A ligand was excluded from the connected paths'
with open(os.path.join(data_path, 'processed', author + '_ligand_connections.json'), 'w') as f:
    json.dump(ligand_connections, f)
    

all_nodes_ = sorted(set(sn_ppis[source_label].tolist() + sn_ppis[target_label].tolist()))
print('Connected paths filtered the network from {} to {} interactions, leaving a total of {} nodes'.format(n_og, sn_ppis.shape[0], len(all_nodes_)))
# print('{} of {} TFs are retained in the final signaling network'.format(len(ligand_connections['IFNB1']), 
#                                                                         len(tf_labels)))
ppi_nodes = set(sn_ppis[source_label].tolist() + sn_ppis[target_label].tolist())
n_tf_intersect = len(ppi_nodes.intersection(tf_labels))
print('{} of {} TFs are present in upon filtering for connected paths'.format(n_tf_intersect, 
                                                                          n_tf_intersect_, 
                                                                         ))


Connected paths filtered the network from 26266 to 17597 interactions, leaving a total of 2504 nodes
380 of 423 TFs are present in upon filtering for connected paths


Format network as needed for model building:

In [14]:
sn_ppis = pn.format_network(sn_ppis, weight_label, stimulation_label, inhibition_label) 

In [15]:
sn_ppis[[source_label, target_label, weight_label, stimulation_label, inhibition_label]].head()

,source_genesymbol,target_genesymbol,mode_of_action,consensus_stimulation,consensus_inhibition
0,CALM1,TRPC1,-1.0,False,True
1,CALM3,TRPC1,-1.0,False,True
2,CALM2,TRPC1,-1.0,False,True
3,CAV1,TRPC1,1.0,True,False
6,ITPR2,TRPC1,1.0,True,False


In [16]:
sn_ppis.to_csv(os.path.join(data_path, 'processed', author + '_sn_ppis.csv'))